# GISLR — Landmark-Subset Discriminability Comparison

**What this notebook does.** Scores every landmark subset registered in
`modules/dataset/landmark/subsets.py` (`FULL_543`, `FP_118`, `ME_126`, `ME_132`,
`HANDS_42`, `HANDS_POSE_50`) on how much **class information** its landmarks carry.
It is the follow-up instrument to the motion-energy analysis
(`gislr.0.dataset.motion-energy.ipynb`, `docs/2026-07-15.md`), which measures
*movement* but is blind to *discriminativeness* — a landmark that moves identically
in every sign has high motion energy and zero class information. **Standalone
diagnostic, not a pipeline stage** (TODO §3.0).

## Methods (TODO §3)

Everything works on **per-video, per-landmark descriptors** — 6 summary statistics
per landmark per video, xy only (the z channel is ~92% noise for pose,
`docs/2026-07-15.md` §3.3): `rms_speed_xy` (same Savitzky-Golay + valid-transition
RMS as the motion-energy notebook, for continuity), `x_mean`, `y_mean`, `x_std`,
`y_std`, `detection_rate`.

| instrument | level | what it measures |
|---|---|---|
| within-class consistency | landmark | CV of `rms_speed_xy` + positional spread across videos of ONE class — how repeatable a landmark's behaviour is inside a class (scope A) |
| ANOVA F-ratio (`f_classif`) | landmark | between-class / within-class variance per descriptor; landmark score = max over its 6 descriptors — the TODO §3 "ANOVA-style variance ratio" |
| mutual information (`mutual_info_classif`) | landmark | non-linear descriptor↔label dependence (scope B only — the kNN estimator is too costly at 94K×3,258) |
| **probe classifier** | **subset** | multinomial logistic regression on the subset's flattened descriptors, stratified 90/10 split seed 42 (at global scope: the canonical 9,448-video val set). **The headline subset score.** |

The probe is deliberately weaker than the GRU — it sees per-video summary
statistics, not trajectories — so it ranks **input information content**, not
achievable model accuracy.

## Scopes (seed 42 everywhere)

1. **Scope A** — 10 random videos of one random sign class → within-class
   consistency only (one class ⇒ discriminability is undefined).
2. **Scope B** — all videos of 10 random classes — the **same 10 signs** as the
   motion-energy per-category sample, for cross-report comparability → F-ratio +
   MI + probe.
3. **Scope C** — global: all 94,477 videos, 250 classes → F-ratio + probe with
   **per-class stats**.

## Artifacts (all under `data/cache/gislr/subset_comparison/`)

| path | content |
|---|---|
| `scope_a_sample.json` / `scope_b_sample.json` | seeded samples (stable across re-runs) |
| `scope_a_within_class/<sign>.parquet` | scope A descriptors |
| `scope_a_consistency.csv` / `scope_a_consistency.png` | per-landmark within-class consistency |
| `scope_b_10class/<sign>.parquet` | scope B descriptors, one file per class |
| `scope_b_landmark_scores.parquet` / `scope_b_subset_scores.csv` | per-landmark F+MI / probe table |
| `scope_c_global/chunk_XXXX.parquet` | global descriptors, 500 videos per chunk |
| `scope_c_landmark_scores.parquet` / `scope_c_subset_scores.csv` | global per-landmark F / probe table |
| `scope_c_per_class/<subset>.csv` | per-class probe recall for every subset |
| `leaderboard.csv` / `subset_scores.json` | combined verdict — fed back into `subsets.py` |
| `<scope>_manifest.json` | resumability manifests |

## Resumability & re-runnability

Descriptor extraction (the only heavy part; the global scope takes ~1h, cf. the
motion-energy global run) goes through the same manifest pattern as the
motion-energy notebook (TODO §1.2): per-unit parquet written **before** the unit
is marked `done`, atomic manifest saves (temp file + `os.replace`), `done`
skipped / `failed` retried on re-run. Every analysis cell reads **only** cached
parquets, so each cell is independently re-runnable once its scope's extraction
cell has completed at least once.

## Design decisions vs TODO §3

- **xy only** — z dropped from all descriptors (z is noise-dominated, 2026-07-15 §3.3).
- **NaN→0 after tensor assembly** — matches the training-cache NaN policy, so the
  probe sees exactly what a model would see.
- **Scope B reuses the motion-energy sign sample** instead of resampling.
- **MI is scope-B only**; the F-ratio is the cross-scope per-landmark instrument.
- Data is downloaded via `kagglehub.competition_download("asl-signs")`
  directly (not `modules.paths`, which resolves every dataset at import time).

In [ ]:
# ============================================================
# Imports & config
# ============================================================
import json
import os
import time
import warnings
from datetime import datetime, timezone
from pathlib import Path

import duckdb
import kagglehub
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.signal import savgol_filter
from scipy.stats import spearmanr
from tqdm.auto import tqdm

from modules.dataset.landmark.subsets import N_LANDMARKS, SUBSETS

DATA_DIR = Path(kagglehub.competition_download("asl-signs"))
from modules.paths import CACHE_DIR

SC_CACHE_DIR = CACHE_DIR / "gislr" / "subset_comparison"
SC_CACHE_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42                # every sampling/split step below uses this seed
SAVGOL_WINDOW = 7        # frames — same smoothing as the motion-energy notebook
SAVGOL_POLYORDER = 2
LOAD_BATCH = 100         # videos per DuckDB read — bounds peak memory
CHUNK_SIZE = 500         # videos per resumable unit in the global scope
N_SCOPE_A_VIDEOS = 10    # scope A: videos of the single sampled class
N_SCOPE_B_CLASSES = 10   # scope B: sampled sign classes
VAL_FRAC = 0.10          # probe val split (global: canonical 90/10 → 9,448 val)
PROBE_MAX_ITER = 200     # logistic-regression lbfgs budget
PROBE_TOL = 1e-3

DESCRIPTORS = ["rms_speed_xy", "x_mean", "y_mean", "x_std", "y_std", "detection_rate"]
TYPE_ROW_OFFSET = {"face": 0, "left_hand": 468, "pose": 489, "right_hand": 522}
TYPE_ORDER = ["pose", "left_hand", "right_hand", "face"]
TYPE_COLORS = {"pose": "tab:blue", "left_hand": "tab:orange",
               "right_hand": "tab:green", "face": "tab:red"}

print("DATA_DIR:", DATA_DIR)
print("cache   :", SC_CACHE_DIR.resolve())
print("subsets :", {n: len(s) for n, s in SUBSETS.items()})

## 1. Reusable core — loading, descriptors, manifests, scoring

Same DuckDB loading layer and manifest-driven resumable driver as the
motion-energy notebook (validated over 249 units / 0 failures), plus this
notebook's own core: `compute_landmark_descriptors` (the per-video ×
per-landmark descriptor extractor every scope runs) and the scoring functions
(`f_ratio_per_landmark`, `mi_per_landmark`, `probe_subset`). These cells are
run-once definitions — parameter iteration happens in the scope cells below.

In [ ]:
# ============================================================
# get_duckdb_conn() + meta — DuckDB loading layer (as in gislr.0.motion-energy)
# ============================================================
def get_duckdb_conn(data_dir: Path) -> duckdb.DuckDBPyConnection:
    """In-memory DuckDB with a `meta` view over train.csv."""
    con = duckdb.connect(database=":memory:")
    root = data_dir.as_posix()
    con.execute(f"""
        CREATE OR REPLACE VIEW meta AS
        SELECT path, participant_id, sequence_id, sign,
               '{root}/' || path AS abs_path
        FROM read_csv_auto('{root}/train.csv')
    """)
    return con


con = get_duckdb_conn(DATA_DIR)
# ORDER BY path: read_csv_auto is parallel and does not guarantee row order —
# a deterministic meta order is what makes the seeded sampling reproducible.
meta = con.execute("SELECT * FROM meta ORDER BY path").df()
print(f"{len(meta):,} videos · {meta['sign'].nunique()} signs · "
      f"{meta['participant_id'].nunique()} participants")


def load_landmarks_for_paths(
    con: duckdb.DuckDBPyConnection,
    paths: list[str],
    data_dir: Path = DATA_DIR,
) -> pd.DataFrame:
    """Load landmark rows (+ sign metadata) for train.csv-relative parquet paths."""
    abs_paths = [(data_dir / p).as_posix() for p in paths]
    return con.execute("""
        SELECT m.sequence_id    AS video_id,
               m.sign           AS sign,
               l.frame, l.type, l.landmark_index, l.x, l.y
        FROM read_parquet($paths, filename=true) l
        JOIN meta m ON replace(l.filename, '\\', '/') = m.abs_path
    """, {"paths": abs_paths}).df()

In [ ]:
# ============================================================
# compute_landmark_descriptors() — the per-(video, landmark) extractor
# ============================================================
def compute_landmark_descriptors(
    df: pd.DataFrame,
    window: int = SAVGOL_WINDOW,
    polyorder: int = SAVGOL_POLYORDER,
) -> pd.DataFrame:
    """6 descriptors per landmark per video, xy only.

    rms_speed_xy follows the motion-energy recipe exactly (reindex to a
    contiguous frame range, linear-interpolate gaps, Savitzky-Golay smooth,
    RMS over transitions whose BOTH endpoints were observed in the raw data);
    positional stats (x/y mean/std) and detection_rate come from the raw
    (pre-interpolation) values.

    Returns tidy long format:
        video_id, sign, row, type, landmark_index,
        rms_speed_xy, x_mean, y_mean, x_std, y_std, detection_rate,
        n_valid_transitions
    """
    out = []
    for (vid, sign), g in df.groupby(["video_id", "sign"], sort=False):
        wide = g.pivot(index="frame", columns=["type", "landmark_index"],
                       values=["x", "y"]).sort_index(axis=1)
        f0, f1 = int(wide.index.min()), int(wide.index.max())
        wide = wide.reindex(np.arange(f0, f1 + 1))
        T = len(wide)
        landmark_cols = wide["x"].columns          # MultiIndex (type, landmark_index)
        L = len(landmark_cols)

        xv = wide["x"].to_numpy()                  # (T, L) raw, NaN where undetected
        yv = wide["y"].to_numpy()
        valid = ~np.isnan(xv)
        with warnings.catch_warnings():            # all-NaN landmarks are expected
            warnings.simplefilter("ignore", category=RuntimeWarning)
            x_mean, y_mean = np.nanmean(xv, axis=0), np.nanmean(yv, axis=0)
            x_std, y_std = np.nanstd(xv, axis=0), np.nanstd(yv, axis=0)

        filled = wide.interpolate(method="linear", limit_direction="both").to_numpy()
        if T >= 3:
            w = min(window, T if T % 2 == 1 else T - 1)
            po = min(polyorder, w - 1)
            allnan = np.isnan(filled).all(axis=0)
            sm = filled.copy()
            sm[:, ~allnan] = savgol_filter(filled[:, ~allnan], w, po, axis=0)
        else:
            sm = filled

        if T >= 2:
            vel = np.diff(sm.reshape(T, 2, L), axis=0)   # (T-1, 2, L)
            speed2 = (vel ** 2).sum(axis=1)              # (T-1, L)
            vt = valid[1:] & valid[:-1]                  # both endpoints observed
            with np.errstate(invalid="ignore", divide="ignore"):
                rms = np.sqrt(np.where(vt, speed2, 0.0).sum(axis=0) / vt.sum(axis=0))
            nvt = vt.sum(axis=0)
        else:
            rms, nvt = np.full(L, np.nan), np.zeros(L, dtype=int)

        types = landmark_cols.get_level_values(0)
        lidx = landmark_cols.get_level_values(1).to_numpy()
        rows = np.array([TYPE_ROW_OFFSET[t] for t in types]) + lidx
        out.append(pd.DataFrame({
            "video_id": vid, "sign": sign,
            "row": rows.astype(np.int16), "type": types, "landmark_index": lidx,
            "rms_speed_xy": rms.astype(np.float32),
            "x_mean": x_mean.astype(np.float32), "y_mean": y_mean.astype(np.float32),
            "x_std": x_std.astype(np.float32), "y_std": y_std.astype(np.float32),
            "detection_rate": valid.mean(axis=0).astype(np.float32),
            "n_valid_transitions": nvt.astype(np.int32),
        }))
    return pd.concat(out, ignore_index=True)


# smoke test — one seeded random video
_smoke_path = meta["path"].sample(1, random_state=SEED).iloc[0]
_smoke = compute_landmark_descriptors(load_landmarks_for_paths(con, [_smoke_path]))
assert len(_smoke) == N_LANDMARKS, f"expected {N_LANDMARKS} rows, got {len(_smoke)}"
assert _smoke["row"].is_unique and _smoke["row"].max() == N_LANDMARKS - 1
_smoke.groupby("type", observed=True)[DESCRIPTORS].mean().round(4)

In [ ]:
# ============================================================
# Manifest helpers + resumable driver (pattern from gislr.0.motion-energy, TODO §1.2)
# ============================================================
def _manifest_path(scope: str) -> Path:
    return SC_CACHE_DIR / f"{scope}_manifest.json"


def _unit_path(scope: str, unit_id: str) -> Path:
    return SC_CACHE_DIR / scope / f"{unit_id}.parquet"


def load_manifest(scope: str) -> dict:
    p = _manifest_path(scope)
    return json.loads(p.read_text()) if p.exists() else {}


def save_manifest(scope: str, manifest: dict) -> None:
    """Atomic write — a crash mid-save can't corrupt the manifest."""
    p = _manifest_path(scope)
    tmp = p.with_suffix(".json.tmp")
    tmp.write_text(json.dumps(manifest, indent=2))
    os.replace(tmp, p)


def process_units(scope: str, unit_ids: list, process_fn, desc: str | None = None) -> dict:
    """Run `process_fn(unit_id) -> DataFrame` over units, resumably.

    Artifact written FIRST, unit marked `done` after; `done` skipped, `failed`
    retried; exceptions recorded in the manifest instead of aborting the run.
    """
    (SC_CACHE_DIR / scope).mkdir(parents=True, exist_ok=True)
    manifest = load_manifest(scope)
    n_skip = n_ok = n_fail = 0
    for uid in tqdm([str(u) for u in unit_ids], desc=desc or scope):
        entry = manifest.get(uid)
        if entry and entry["status"] == "done" and Path(entry["artifact"]).exists():
            n_skip += 1
            continue
        try:
            out = process_fn(uid)
            artifact = _unit_path(scope, uid)
            out.to_parquet(artifact, index=False)
            manifest[uid] = {"status": "done",
                             "timestamp": datetime.now(timezone.utc).isoformat(),
                             "artifact": str(artifact)}
            n_ok += 1
        except Exception as e:
            manifest[uid] = {"status": "failed",
                             "timestamp": datetime.now(timezone.utc).isoformat(),
                             "artifact": None, "error": repr(e)}
            n_fail += 1
        save_manifest(scope, manifest)
    print(f"[{scope}] done={n_ok}  skipped={n_skip}  failed={n_fail}")
    if n_fail:
        print("failed units:",
              {k: v["error"] for k, v in manifest.items() if v["status"] == "failed"})
    return manifest


def load_cached_units(scope: str, unit_ids: list) -> pd.DataFrame:
    """Analysis input: reads ONLY cached per-unit parquets, never raw data."""
    frames = [pd.read_parquet(_unit_path(scope, str(u)))
              for u in unit_ids if _unit_path(scope, str(u)).exists()]
    return pd.concat(frames, ignore_index=True)

In [ ]:
# ============================================================
# Scoring core — descriptor tensor, F-ratio, MI, probe, plots
# ============================================================
from sklearn.feature_selection import f_classif, mutual_info_classif
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

ROW_TYPE = np.empty(N_LANDMARKS, dtype=object)
for _t, _o in TYPE_ROW_OFFSET.items():   # offsets ascend in insertion order
    ROW_TYPE[_o:] = _t


def descriptor_tensor(desc_df: pd.DataFrame) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Long descriptor df → (X (n, 543, 6) float32 NaN→0, y signs, video_ids).

    NaN→0 matches the training-cache NaN policy — the probe sees what a model sees.
    """
    vids = desc_df["video_id"].drop_duplicates().to_numpy()
    pos = {v: k for k, v in enumerate(vids)}
    i = desc_df["video_id"].map(pos).to_numpy()
    j = desc_df["row"].to_numpy()
    X = np.full((len(vids), N_LANDMARKS, len(DESCRIPTORS)), np.nan, dtype=np.float32)
    for k, d in enumerate(DESCRIPTORS):
        X[i, j, k] = desc_df[d].to_numpy(dtype=np.float32)
    y = (desc_df.drop_duplicates("video_id").set_index("video_id")["sign"]
         .reindex(vids).to_numpy())
    return np.nan_to_num(X, nan=0.0), y, vids


def f_ratio_per_landmark(X: np.ndarray, y: np.ndarray) -> pd.DataFrame:
    """ANOVA F per (landmark, descriptor); landmark score = max over descriptors."""
    with warnings.catch_warnings():   # constant features (never-detected) warn
        warnings.simplefilter("ignore")
        F, _ = f_classif(X.reshape(len(X), -1), y)
    F = np.nan_to_num(F.reshape(N_LANDMARKS, len(DESCRIPTORS)), nan=0.0, posinf=0.0)
    out = pd.DataFrame(F, columns=[f"F_{d}" for d in DESCRIPTORS])
    out.insert(0, "row", np.arange(N_LANDMARKS))
    out.insert(1, "type", ROW_TYPE)
    out["F_max"] = F.max(axis=1)
    return out


def mi_per_landmark(X: np.ndarray, y: np.ndarray, seed: int = SEED) -> pd.DataFrame:
    """Mutual information per (landmark, descriptor); scope-B-scale only."""
    mi = mutual_info_classif(X.reshape(len(X), -1), y, random_state=seed, n_jobs=8)
    MI = mi.reshape(N_LANDMARKS, len(DESCRIPTORS))
    out = pd.DataFrame(MI, columns=[f"MI_{d}" for d in DESCRIPTORS])
    out.insert(0, "row", np.arange(N_LANDMARKS))
    out["MI_max"] = MI.max(axis=1)
    return out


def probe_subset(X: np.ndarray, y: np.ndarray, subset,
                 max_iter: int = PROBE_MAX_ITER) -> tuple[dict, pd.DataFrame]:
    """Multinomial logistic probe on a subset's flattened descriptors.

    Stratified 90/10 split, seed 42 — at global scope this is the canonical
    eval split (9,448 val videos). Returns (summary dict, per-class recall).
    """
    Xs = X[:, subset.array, :].reshape(len(X), -1)
    Xtr, Xva, ytr, yva = train_test_split(
        Xs, y, test_size=VAL_FRAC, stratify=y, random_state=SEED)
    scaler = StandardScaler().fit(Xtr)
    clf = LogisticRegression(max_iter=max_iter, tol=PROBE_TOL)
    t0 = time.time()
    clf.fit(scaler.transform(Xtr), ytr)
    pred = clf.predict(scaler.transform(Xva))
    per_class = (pd.DataFrame({"sign": yva, "correct": pred == yva})
                 .groupby("sign")["correct"].agg(["mean", "size"])
                 .rename(columns={"mean": "recall", "size": "n_val"}))
    summary = {"subset": subset.name, "n_landmarks": len(subset),
               "probe_acc": round(float(accuracy_score(yva, pred)), 4),
               "probe_macro": round(float(balanced_accuracy_score(yva, pred)), 4),
               "n_val": len(yva), "fit_s": round(time.time() - t0, 1),
               "n_iter": int(np.max(clf.n_iter_))}
    return summary, per_class


def subset_median_stat(landmark_df: pd.DataFrame, col: str) -> dict:
    """Median per-landmark score over each subset's member landmarks."""
    s = landmark_df.set_index("row")[col]
    return {name: float(s.loc[list(sub.indices)].median())
            for name, sub in SUBSETS.items()}


def plot_landmark_scores(scores, title: str, ylabel: str,
                         logy: bool = True, fname: str | None = None):
    fig, ax = plt.subplots(figsize=(14, 4))
    rows = np.arange(N_LANDMARKS)
    for t in TYPE_ORDER:
        m = ROW_TYPE == t
        ax.scatter(rows[m], np.asarray(scores)[m], s=6, color=TYPE_COLORS[t], label=t)
    if logy:
        ax.set_yscale("log")
    ax.set_xlabel("holistic row")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.legend(markerscale=2, ncol=4)
    fig.tight_layout()
    if fname:
        fig.savefig(SC_CACHE_DIR / fname, dpi=120)
    return fig

## 2. Scope A — within-class consistency: 10 videos of one class (TODO §3.0)

One random sign class (seed 42), 10 random videos of it. With a single class
there is no discriminability to measure — this scope answers the *within-class*
half of TODO §3: which landmarks behave **repeatably** across renditions of the
same sign. Metrics per landmark, across the 10 videos:

- `cv_speed` — coefficient of variation of `rms_speed_xy` (lower = the landmark
  moves the same amount in every rendition);
- `pos_spread` — std across videos of the video-mean position (lower = the
  landmark sits in the same place every time);
- `det_mean` — mean detection rate.

Interpretation caveat: **n = 10 videos of 1 class** — a direction check to
contextualize the scope B/C rankings, not a standalone verdict.

In [ ]:
# ============================================================
# Scope A — sample, extract (resumable), score consistency
# ============================================================
SCOPE_A = "scope_a_within_class"

_sample_file_a = SC_CACHE_DIR / "scope_a_sample.json"
if _sample_file_a.exists():
    scope_a = json.loads(_sample_file_a.read_text())
else:
    rng = np.random.default_rng(SEED)
    _sign = str(rng.choice(sorted(meta["sign"].unique())))
    _paths = (meta.loc[meta["sign"] == _sign, "path"]
              .sample(N_SCOPE_A_VIDEOS, random_state=SEED).tolist())
    scope_a = {"seed": SEED, "sign": _sign, "paths": _paths}
    _sample_file_a.write_text(json.dumps(scope_a, indent=2))
print(f"scope A class: {scope_a['sign']!r} · {len(scope_a['paths'])} videos")

process_units(
    SCOPE_A, [scope_a["sign"]],
    lambda uid: compute_landmark_descriptors(
        load_landmarks_for_paths(con, scope_a["paths"])),
    desc="scope A")

desc_a = load_cached_units(SCOPE_A, [scope_a["sign"]])
g = desc_a.groupby("row")
with warnings.catch_warnings():
    warnings.simplefilter("ignore", category=RuntimeWarning)
    cons = pd.DataFrame({
        "cv_speed": g["rms_speed_xy"].std() / g["rms_speed_xy"].mean().abs(),
        "pos_spread": np.sqrt(g["x_mean"].var() + g["y_mean"].var()),
        "det_mean": g["detection_rate"].mean(),
    }).reset_index()
cons["type"] = ROW_TYPE[cons["row"].to_numpy()]
cons.to_csv(SC_CACHE_DIR / "scope_a_consistency.csv", index=False)

_c = cons.set_index("row")
tbl_a = pd.DataFrame([{
    "subset": name, "n_landmarks": len(s),
    "median_cv_speed": round(float(_c.loc[list(s.indices), "cv_speed"].median()), 4),
    "median_pos_spread": round(float(_c.loc[list(s.indices), "pos_spread"].median()), 4),
    "mean_detection": round(float(_c.loc[list(s.indices), "det_mean"].mean()), 3),
} for name, s in SUBSETS.items()]).sort_values("median_cv_speed")

fig = plot_landmark_scores(
    cons.sort_values("row")["cv_speed"].to_numpy(),
    f"Within-class consistency — CV of rms_speed_xy across {len(scope_a['paths'])} "
    f"videos of {scope_a['sign']!r} (lower = more consistent)",
    "CV of rms_speed_xy", logy=False, fname="scope_a_consistency.png")
plt.show()
tbl_a

## 3. Scope B — all videos of 10 random classes (TODO §3.0)

The same 10 signs as the motion-energy per-category sample (seed 42 →
`beside, bird, cow, hat, hate, old, pizza, ride, table, yourself`), all their
videos (~3,750). Small enough that all three discriminability instruments run —
per-landmark **F-ratio**, per-landmark **MI**, and the **probe** per subset —
and big enough for stable estimates (~375 videos/class). One resumable unit per
class; batched loads of 100 videos bound peak memory.

In [ ]:
# ============================================================
# Scope B — descriptor extraction (resumable, one unit per class)
# ============================================================
SCOPE_B = "scope_b_10class"

_sample_file_b = SC_CACHE_DIR / "scope_b_sample.json"
_me_sample = CACHE_DIR / "gislr" / "motion_analysis" / "per_category_sample.json"
if _sample_file_b.exists():
    scope_b_signs = json.loads(_sample_file_b.read_text())["signs"]
elif _me_sample.exists():   # reuse the motion-energy sample for cross-report comparability
    scope_b_signs = json.loads(_me_sample.read_text())["signs"]
    _sample_file_b.write_text(json.dumps(
        {"seed": SEED, "source": str(_me_sample), "signs": scope_b_signs}, indent=2))
else:
    rng = np.random.default_rng(SEED)
    scope_b_signs = sorted(rng.choice(sorted(meta["sign"].unique()),
                                      N_SCOPE_B_CLASSES, replace=False).tolist())
    _sample_file_b.write_text(json.dumps(
        {"seed": SEED, "source": "fresh sample", "signs": scope_b_signs}, indent=2))
print("scope B signs:", scope_b_signs)


def _process_sign(sign: str) -> pd.DataFrame:
    paths = meta.loc[meta["sign"] == sign, "path"].tolist()
    return pd.concat(
        [compute_landmark_descriptors(load_landmarks_for_paths(con, paths[i:i + LOAD_BATCH]))
         for i in range(0, len(paths), LOAD_BATCH)],
        ignore_index=True)


process_units(SCOPE_B, scope_b_signs, _process_sign, desc="scope B classes")

In [ ]:
# ============================================================
# Scope B — analysis: F-ratio + MI per landmark, probe per subset
# ============================================================
scope_b_signs = json.loads((SC_CACHE_DIR / "scope_b_sample.json").read_text())["signs"]
desc_b = load_cached_units("scope_b_10class", scope_b_signs)
Xb, yb, _ = descriptor_tensor(desc_b)
print(f"scope B tensor: {Xb.shape}  ({len(np.unique(yb))} classes)")

fb = f_ratio_per_landmark(Xb, yb)
_t0 = time.time()
mib = mi_per_landmark(Xb, yb)
print(f"MI computed in {time.time() - _t0:.0f}s")
lm_b = fb.merge(mib, on="row")
lm_b.to_parquet(SC_CACHE_DIR / "scope_b_landmark_scores.parquet", index=False)

rho_fm = spearmanr(lm_b["F_max"], lm_b["MI_max"]).statistic
print(f"method agreement (Spearman F_max vs MI_max, n=543): {rho_fm:.3f}")

fig = plot_landmark_scores(lm_b["F_max"].to_numpy(),
                           "Scope B — per-landmark ANOVA F (max over descriptors), 10 classes",
                           "F_max (log)", fname="scope_b_f_ratio.png")
plt.show()
fig = plot_landmark_scores(lm_b["MI_max"].to_numpy(),
                           "Scope B — per-landmark mutual information (max over descriptors)",
                           "MI_max", logy=False, fname="scope_b_mi.png")
plt.show()

rows_b = []
for name, s in SUBSETS.items():
    summ, _ = probe_subset(Xb, yb, s)
    print(summ)
    rows_b.append(summ)
tbl_b = pd.DataFrame(rows_b).sort_values("probe_macro", ascending=False)
tbl_b["median_F"] = tbl_b["subset"].map(subset_median_stat(lm_b, "F_max")).round(1)
tbl_b["median_MI"] = tbl_b["subset"].map(subset_median_stat(lm_b, "MI_max")).round(4)
tbl_b.to_csv(SC_CACHE_DIR / "scope_b_subset_scores.csv", index=False)
tbl_b

## 4. Scope C — global: all 94,477 videos, 250 classes (TODO §3.0)

The full dataset in 189 resumable chunks of ≤500 videos (same chunking as the
motion-energy global scope; expect ~1h for a cold run — re-runs skip `done`
chunks). Analysis: global per-landmark **F-ratio**, and the **probe** per subset
on the canonical stratified 90/10 split (seed 42, 9,448 val videos — the split
every GISLR run is compared on) with **per-class recall** written per subset to
`scope_c_per_class/<subset>.csv`. MI is skipped at this scale (see title cell).

In [ ]:
# ============================================================
# Scope C — descriptor extraction (189 resumable chunks — the ~1h cell)
# ============================================================
SCOPE_C = "scope_c_global"

_all_paths = meta.sort_values("path")["path"].tolist()   # deterministic chunking
chunks = {f"chunk_{i:04d}": _all_paths[i * CHUNK_SIZE:(i + 1) * CHUNK_SIZE]
          for i in range((len(_all_paths) + CHUNK_SIZE - 1) // CHUNK_SIZE)}
print(f"{len(chunks)} chunks of ≤{CHUNK_SIZE} videos")


def _process_chunk(uid: str) -> pd.DataFrame:
    paths = chunks[uid]
    return pd.concat(
        [compute_landmark_descriptors(load_landmarks_for_paths(con, paths[i:i + LOAD_BATCH]))
         for i in range(0, len(paths), LOAD_BATCH)],
        ignore_index=True)


manifest_c = process_units(SCOPE_C, list(chunks), _process_chunk, desc="global chunks")

In [ ]:
# ============================================================
# Scope C — analysis: global F-ratio + probe per subset with per-class stats
# ============================================================
SCOPE_C = "scope_c_global"
_manifest = json.loads((SC_CACHE_DIR / f"{SCOPE_C}_manifest.json").read_text())
chunk_ids = sorted(k for k, v in _manifest.items() if v["status"] == "done")
assert len(chunk_ids) == len(_manifest), \
    f"{len(_manifest) - len(chunk_ids)} chunks not done — re-run the extraction cell"

# incremental tensor assembly — never concatenates the long dataframes
_Xs, _ys = [], []
for uid in tqdm(chunk_ids, desc="load chunk tensors"):
    X_, y_, _ = descriptor_tensor(pd.read_parquet(_unit_path(SCOPE_C, uid)))
    _Xs.append(X_)
    _ys.append(y_)
Xg, yg = np.concatenate(_Xs), np.concatenate(_ys)
del _Xs, _ys
print(f"global tensor: {Xg.shape} = {Xg.nbytes / 1e9:.2f} GB · {len(np.unique(yg))} classes")

fg = f_ratio_per_landmark(Xg, yg)
fg.to_parquet(SC_CACHE_DIR / "scope_c_landmark_scores.parquet", index=False)
fig = plot_landmark_scores(fg["F_max"].to_numpy(),
                           "Global — per-landmark ANOVA F (max over descriptors), 250 classes",
                           "F_max (log)", fname="scope_c_f_ratio.png")
plt.show()

(SC_CACHE_DIR / "scope_c_per_class").mkdir(exist_ok=True)
rows_c = []
for name, s in SUBSETS.items():
    summ, per_class = probe_subset(Xg, yg, s)
    per_class.round(4).to_csv(SC_CACHE_DIR / "scope_c_per_class" / f"{name}.csv")
    print(summ)
    rows_c.append(summ)
tbl_c = pd.DataFrame(rows_c).sort_values("probe_macro", ascending=False)
tbl_c["median_F"] = tbl_c["subset"].map(subset_median_stat(fg, "F_max")).round(1)
tbl_c.to_csv(SC_CACHE_DIR / "scope_c_subset_scores.csv", index=False)
assert tbl_c["n_val"].iloc[0] == 9448, "canonical split should yield 9,448 val videos"
tbl_c

## 5. Verdict — combined leaderboard & cross-method agreement

Assembles the final subset leaderboard from the cached scope artifacts only
(re-runnable without touching raw data): probe accuracy at scopes B and C
(headline = global macro), median member-landmark F, scope A consistency.
Also checks method agreement — F vs MI, scope B vs scope C F-ranks, F vs the
motion-energy global RMS ranking — and externally validates the probe by
correlating its per-class recall (ME_126) against the trained ME-126 GRU's
per-class accuracy. Writes `leaderboard.csv` + `subset_scores.json`, whose
`probe_acc_global` / `probe_macro_global` values are copied into
`modules/dataset/landmark/subsets.py`.

In [ ]:
# ============================================================
# Verdict — leaderboard, agreement checks, exports (reads cache only)
# ============================================================
tbl_b = pd.read_csv(SC_CACHE_DIR / "scope_b_subset_scores.csv")
tbl_c = pd.read_csv(SC_CACHE_DIR / "scope_c_subset_scores.csv")
cons = pd.read_csv(SC_CACHE_DIR / "scope_a_consistency.csv")
lm_b = pd.read_parquet(SC_CACHE_DIR / "scope_b_landmark_scores.parquet")
fg = pd.read_parquet(SC_CACHE_DIR / "scope_c_landmark_scores.parquet")

lb = (tbl_c[["subset", "n_landmarks", "probe_acc", "probe_macro", "median_F"]]
      .rename(columns={"probe_acc": "probe_acc_global",
                       "probe_macro": "probe_macro_global",
                       "median_F": "median_F_global"})
      .merge(tbl_b[["subset", "probe_acc", "probe_macro"]]
             .rename(columns={"probe_acc": "probe_acc_10class",
                              "probe_macro": "probe_macro_10class"}), on="subset"))
_c = cons.set_index("row")
lb["median_cv_speed_1class"] = lb["subset"].map(
    {name: round(float(_c.loc[list(s.indices), "cv_speed"].median()), 4)
     for name, s in SUBSETS.items()})
lb = lb.sort_values("probe_macro_global", ascending=False).reset_index(drop=True)
lb.to_csv(SC_CACHE_DIR / "leaderboard.csv", index=False)
(SC_CACHE_DIR / "subset_scores.json").write_text(
    json.dumps(lb.set_index("subset").to_dict(orient="index"), indent=2))

winner = lb.iloc[0]
print(f"WINNER: {winner['subset']} — global probe {winner['probe_acc_global']:.1%} "
      f"(macro {winner['probe_macro_global']:.1%}) with {winner['n_landmarks']} landmarks\n")

# --- method agreement -------------------------------------------------------
agree = {
    "F_max: scope B vs scope C (n=543)":
        spearmanr(lm_b["F_max"], fg["F_max"]).statistic,
    "scope B: F_max vs MI_max (n=543)":
        spearmanr(lm_b["F_max"], lm_b["MI_max"]).statistic,
}
_me_global = CACHE_DIR / "gislr" / "motion_analysis" / "global" / "summary.parquet"
if _me_global.exists():
    me = pd.read_parquet(_me_global)
    me["row"] = me["type"].map(TYPE_ROW_OFFSET) + me["landmark_index"]
    _j = fg.merge(me[["row", "rms_speed_mean"]], on="row")
    agree["global F_max vs motion-energy RMS (n=543)"] = \
        spearmanr(_j["F_max"], _j["rms_speed_mean"]).statistic
for k, v in agree.items():
    print(f"rank agreement — {k}: rho = {v:.3f}")

# --- external validation: probe per-class recall vs trained GRU (ME_126) ----
_gru_csv = Path("models/gislr/gru/20260715-190729/cache/per_class_accuracy.csv")
_probe_csv = SC_CACHE_DIR / "scope_c_per_class" / "ME_126.csv"
if _gru_csv.exists() and _probe_csv.exists():
    _cmp = pd.read_csv(_probe_csv).merge(pd.read_csv(_gru_csv), on="sign")
    rho = spearmanr(_cmp["recall"], _cmp["accuracy"]).statistic
    print(f"\nprobe per-class recall vs trained ME-126 GRU per-class accuracy: "
          f"rho = {rho:.3f} (n={len(_cmp)} classes)")

# --- leaderboard figure ------------------------------------------------------
fig, ax = plt.subplots(figsize=(8, 4))
_ypos = np.arange(len(lb))[::-1]
ax.barh(_ypos, lb["probe_macro_global"], color="tab:blue", label="global (250 classes)")
ax.barh(_ypos, lb["probe_macro_10class"], height=0.4, color="tab:orange",
        label="10-class (scope B)")
ax.set_yticks(_ypos, [f"{r.subset} ({r.n_landmarks})" for r in lb.itertuples()])
ax.set_xlabel("probe balanced accuracy")
ax.set_title("Landmark-subset probe leaderboard")
ax.legend()
fig.tight_layout()
fig.savefig(SC_CACHE_DIR / "leaderboard.png", dpi=120)
plt.show()

# worst 10 classes under the winning subset
_worst = (pd.read_csv(SC_CACHE_DIR / "scope_c_per_class" / f"{winner['subset']}.csv")
          .sort_values("recall").head(10))
print(f"\nworst 10 classes under {winner['subset']}:")
print(_worst.to_string(index=False))
lb

## Caveats & what this notebook deliberately does NOT do

- **The probe ranks input information, not achievable accuracy.** It sees 6
  summary statistics per landmark, not trajectories — probe numbers are far
  below GRU numbers by construction and only their *ordering across subsets*
  matters. The definitive comparison stays the all-else-identical GRU training
  ablations (TODO §3.1).
- **Descriptors are correlated across landmarks** (e.g. all face landmarks move
  with the head), so per-landmark F/MI overstate *unique* contribution;
  redundancy is exactly what the subset-level probe accounts for.
- **Scope A is one class, 10 videos** — a consistency direction-check, not a
  significance test.
- No gradient saliency / SHAP (needs a trained model — TODO §3), no per-landmark
  greedy selection (combinatorial; the registry subsets are hypothesis-driven),
  no new subset invention here — if the rankings suggest one, it gets added to
  `subsets.py` and trained per TODO §3.1.
- Scores were written back into `modules/dataset/landmark/subsets.py`
  (`probe_acc_global`) after the executed run; the dated write-up lives in
  `docs/2026-07-16.md`.